# FactLens — Live Demo
**Fake News Detection + Political Bias Classification**

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install gradio vaderSentiment -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.2 MB/s eta 0:00:00


In [3]:
import pickle
import numpy as np
import scipy.sparse as sp
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR1  = "/content/drive/MyDrive/FactLens_Group9/data"
ARTIFACTS2 = "/content/drive/MyDrive/FactLens_Group9/data2/artifacts"

# ── Load Dataset 1 (Fake vs Real) ──────────────────────────────────────────
with open(f"{DATA_DIR1}/logistic_regression_model.pkl", "rb") as f:
    lr_d1 = pickle.load(f)
with open(f"{DATA_DIR1}/tfidf_vectorizer.pkl", "rb") as f:
    vec_d1 = pickle.load(f)

# ── Load Dataset 2 (Left vs Right) ─────────────────────────────────────────
with open(f"{ARTIFACTS2}/left_right_logreg.pkl", "rb") as f:
    lr_d2 = pickle.load(f)
with open(f"{ARTIFACTS2}/tfidf_vectorizer.pkl", "rb") as f:
    vec_d2 = pickle.load(f)
with open(f"{ARTIFACTS2}/label_encoder.pkl", "rb") as f:
    le_d2 = pickle.load(f)

analyzer = SentimentIntensityAnalyzer()

# Pre-build coefficient lookup for top word explanations
d1_feature_names = vec_d1.get_feature_names_out()
d1_coefs         = lr_d1.coef_[0]
d2_feature_names = vec_d2.get_feature_names_out()
d2_coefs         = lr_d2.coef_[0]

print("Models loaded successfully.")

Models loaded successfully.


In [4]:
def get_top_words(text, vectorizer, feature_names, coefs, label_is_positive, n=6):
    """Return top words pushing toward the predicted class."""
    X = vectorizer.transform([text])
    nonzero_idx = X.nonzero()[1]
    contributions = []
    for idx in nonzero_idx:
        word  = feature_names[idx]
        coef  = coefs[idx]
        tfidf = X[0, idx]
        score = float(coef * tfidf)
        contributions.append((word, score))
    contributions.sort(key=lambda x: x[1], reverse=label_is_positive)
    top = contributions[:n]
    return top


def classify(text):
    if not text or len(text.strip()) < 50:
        return "Please paste a longer article (at least a few sentences)."

    # ── D1: Fake vs Real ───────────────────────────────────────────────────
    X1   = vec_d1.transform([text])
    prob1 = lr_d1.predict_proba(X1)[0]
    # D1: class 0 = Fake, class 1 = Real (standard sklearn convention)
    # But the model was trained with label 1 = Real, so check:
    classes_d1 = lr_d1.classes_
    real_idx   = list(classes_d1).index(1) if 1 in classes_d1 else 1
    fake_idx   = 1 - real_idx
    p_fake = prob1[fake_idx]
    p_real = prob1[real_idx]
    label_d1 = "FAKE" if p_fake > p_real else "REAL"
    conf_d1  = max(p_fake, p_real)
    is_fake  = label_d1 == "FAKE"
    top_words_d1 = get_top_words(text, vec_d1, d1_feature_names, d1_coefs,
                                  label_is_positive=is_fake)

    # ── D2: Left vs Right ──────────────────────────────────────────────────
    sentiment    = analyzer.polarity_scores(text[:500])["compound"]
    subjectivity = abs(sentiment)
    exclaim      = text.count("!")
    question     = text.count("?")
    extra        = np.array([[sentiment, subjectivity, 0.0, exclaim, question]])

    X2_tfidf = vec_d2.transform([text])
    X2       = sp.hstack([X2_tfidf, sp.csr_matrix(extra)])
    prob2    = lr_d2.predict_proba(X2)[0]
    classes_d2 = lr_d2.classes_   # 0=Left, 1=Right
    p_left  = prob2[0]
    p_right = prob2[1]
    label_d2 = "RIGHT" if p_right > p_left else "LEFT"
    conf_d2  = max(p_left, p_right)
    is_right = label_d2 == "RIGHT"
    top_words_d2 = get_top_words(text, vec_d2, d2_feature_names, d2_coefs,
                                  label_is_positive=is_right)

    # ── Format output ──────────────────────────────────────────────────────
    bar_d1 = "█" * int(conf_d1 * 20) + "░" * (20 - int(conf_d1 * 20))
    bar_d2 = "█" * int(conf_d2 * 20) + "░" * (20 - int(conf_d2 * 20))

    words_d1_str = "  ".join([f"{w} ({s:+.2f})" for w, s in top_words_d1])
    words_d2_str = "  ".join([f"{w} ({s:+.2f})" for w, s in top_words_d2])

    result = f"""
╔══════════════════════════════════════════════════════╗
║           FACTLENS CLASSIFICATION RESULTS            ║
╚══════════════════════════════════════════════════════╝

📰  FAKE vs REAL NEWS
    Verdict   : {'🔴 FAKE NEWS' if is_fake else '🟢 REAL NEWS'}
    Confidence: {conf_d1:.1%}  [{bar_d1}]
    Key words : {words_d1_str}

🏛️  POLITICAL BIAS
    Verdict   : {'🔵 LEFT-LEANING' if not is_right else '🔴 RIGHT-LEANING'}
    Confidence: {conf_d2:.1%}  [{bar_d2}]
    Key words : {words_d2_str}

──────────────────────────────────────────────────────
  Note: Political bias is only meaningful for articles
  with clear political content.
"""
    return result


print("Classifier ready.")

Classifier ready.


In [5]:
import gradio as gr

EXAMPLES = [
    ["BREAKING: Scientists confirm that drinking bleach cures cancer. The mainstream media is hiding this from you. Share before they delete it! President Obama was secretly behind the coverup along with George Soros and the deep state. Thousands of doctors have been silenced. Wake up people!!!"],
    ["The Federal Reserve raised interest rates by a quarter percentage point on Wednesday, its tenth increase since March 2022, as policymakers continued their fight against inflation. Fed Chair Jerome Powell said officials would monitor incoming data carefully before deciding whether additional increases are needed."],
    ["The Republican-led House passed a sweeping tax cut bill along party lines Thursday, slashing the corporate tax rate and reducing levies on the wealthy. Democrats blasted the measure as a giveaway to the rich that would balloon the deficit and hurt working families."],
    ["Senate Republicans blocked Democratic efforts to expand background checks for gun purchases, with GOP leaders arguing the legislation would infringe on Second Amendment rights and do little to prevent crime. The National Rifle Association applauded the vote."],
]

with gr.Blocks(theme=gr.themes.Soft(), title="FactLens") as demo:

    gr.HTML("""
    <div style='text-align:center; padding: 20px 0 10px;'>
        <h1 style='font-size:2.5em; font-weight:700; margin:0;'>🔍 FactLens</h1>
        <p style='font-size:1.1em; color:#555; margin-top:6px;'>
            Fake News Detection &amp; Political Bias Classification
        </p>
        <p style='font-size:0.9em; color:#888;'>CAP 5610 · Group 9 · FIU</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            text_input = gr.Textbox(
                label="Paste an article or news text here",
                placeholder="Paste any news article...",
                lines=14,
                max_lines=20,
            )
            classify_btn = gr.Button("🔍  Analyze", variant="primary", size="lg")

            gr.Examples(
                examples=EXAMPLES,
                inputs=text_input,
                label="Try these examples",
            )

        with gr.Column(scale=1):
            output = gr.Textbox(
                label="Results",
                lines=18,
                interactive=False,
                show_copy_button=True,
            )

    classify_btn.click(fn=classify, inputs=text_input, outputs=output)
    text_input.submit(fn=classify, inputs=text_input, outputs=output)

    gr.HTML("""
    <div style='text-align:center; padding:16px 0 4px; color:#aaa; font-size:0.8em;'>
        Models: Logistic Regression + TF-IDF &nbsp;|&nbsp;
        D1: 97.5% accuracy &nbsp;|&nbsp; D2: 87.1% accuracy
    </div>
    """)

demo.launch(share=True, debug=False)

/tmp/ipykernel_6951/3325483355.py:10: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="FactLens") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://765c19bd6f7d35110c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
